In [1]:
import tensorflow as tf
import pandas as pd

print("TensorFlow:", tf.__version__)

TensorFlow: 2.21.0


In [2]:
inputs = tf.constant([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]
], dtype=tf.float32)

labels = tf.constant([
    [0.0],
    [1.0],
    [1.0],
    [0.0]
], dtype=tf.float32)

pd.DataFrame({
    "x1": inputs[:, 0].numpy(),
    "x2": inputs[:, 1].numpy(),
    "y_true": labels[:, 0].numpy()
})

,x1,x2,y_true
0,0.0,0.0,0.0
1,0.0,1.0,1.0
2,1.0,0.0,1.0
3,1.0,1.0,0.0


In [3]:
LR = 0.3
H = 4
EPOCHS = 6000

In [4]:
def sigmoid_approx(x):
    return tf.clip_by_value(0.5 + 0.125 * x, 0.0, 1.0)

def sigmoid_derivative(a):
    return a * (1.0 - a)

In [5]:
W1_init = tf.constant([
    [ 0.2, -0.1,  0.4,  0.1],
    [-0.3,  0.2,  0.1, -0.2]
], dtype=tf.float32)

b1_init = tf.constant([0.0, 0.0, 0.0, 0.0], dtype=tf.float32)

W2_init = tf.constant([0.3, -0.4, 0.2, 0.1], dtype=tf.float32)

b2_init = tf.constant(0.05, dtype=tf.float32)

print("W1_init:\n", W1_init.numpy())
print("b1_init:", b1_init.numpy())
print("W2_init:", W2_init.numpy())
print("b2_init:", b2_init.numpy())

W1_init:
 [[ 0.2 -0.1  0.4  0.1]
 [-0.3  0.2  0.1 -0.2]]
b1_init: [0. 0. 0. 0.]
W2_init: [ 0.3 -0.4  0.2  0.1]
b2_init: 0.05


In [6]:
def forward(x, W1, b1, W2, b2):
    z1 = tf.matmul(tf.reshape(x, [1, 2]), W1) + tf.reshape(b1, [1, H])
    a1 = sigmoid_approx(z1)
    s = tf.reduce_sum(a1 * tf.reshape(W2, [1, H]), axis=1) + b2
    a2 = sigmoid_approx(s)
    pred = tf.cast(a2 > 0.5, tf.int32)
    return z1, a1, s, a2, pred

In [7]:
x = tf.constant([1.0, 0.0], dtype=tf.float32)

z1, a1, s, a2, pred = forward(x, W1_init, b1_init, W2_init, b2_init)

print("z1   =", z1.numpy())
print("a1   =", a1.numpy())
print("sum  =", s.numpy())
print("a2   =", a2.numpy())
print("pred =", pred.numpy())

z1   = [[ 0.2 -0.1  0.4  0.1]]
a1   = [[0.525  0.4875 0.55   0.5125]]
sum  = [0.17375]
a2   = [0.52171874]
pred = [1]


In [8]:
def train_one(x, y_true, W1, b1, W2, b2, lr=0.3):
    z1, a1, s, a2, pred = forward(x, W1, b1, W2, b2)

    a2_scalar = a2[0]
    d2 = (a2_scalar - y_true) * sigmoid_derivative(a2_scalar)
    d1 = d2 * W2 * sigmoid_derivative(tf.reshape(a1, [H]))

    W2_new = W2 - lr * (tf.reshape(a1, [H]) * d2)
    b2_new = b2 - lr * d2

    W1_new = tf.identity(W1)
    W1_new = tf.tensor_scatter_nd_sub(W1_new, [[0, j] for j in range(H)], lr * (x[0] * d1))
    W1_new = tf.tensor_scatter_nd_sub(W1_new, [[1, j] for j in range(H)], lr * (x[1] * d1))

    b1_new = b1 - lr * d1

    return {
        "z1": z1,
        "a1": a1,
        "sum": s,
        "a2": a2_scalar,
        "pred": pred[0],
        "d2": d2,
        "d1": d1,
        "W1_new": W1_new,
        "b1_new": b1_new,
        "W2_new": W2_new,
        "b2_new": b2_new
    }

In [9]:
x = tf.constant([1.0, 0.0], dtype=tf.float32)
y_true = tf.constant(1.0, dtype=tf.float32)

step_result = train_one(x, y_true, W1_init, b1_init, W2_init, b2_init, lr=LR)

print("a2      =", step_result["a2"].numpy())
print("pred    =", step_result["pred"].numpy())
print("d2      =", step_result["d2"].numpy())
print("W1_00   =", step_result["W1_new"][0,0].numpy())
print("W1_10   =", step_result["W1_new"][1,0].numpy())
print("b1_0    =", step_result["b1_new"][0].numpy())
print("W2_0    =", step_result["W2_new"][0].numpy())
print("b2      =", step_result["b2_new"].numpy())

a2      = 0.52171874
pred    = 1
d2      = -0.11934471
W1_00   = 0.20267855
W1_10   = -0.3
b1_0    = 0.002678543
W2_0    = 0.3187968
b2      = 0.08580342


In [10]:
def train_epochs(W1, b1, W2, b2, epochs=6000, lr=0.3):
    W1 = tf.identity(W1)
    b1 = tf.identity(b1)
    W2 = tf.identity(W2)
    b2 = tf.identity(b2)

    history = []

    for e in range(epochs):
        for i in range(4):
            x = inputs[i]
            y = labels[i, 0]
            out = train_one(x, y, W1, b1, W2, b2, lr)
            W1 = out["W1_new"]
            b1 = out["b1_new"]
            W2 = out["W2_new"]
            b2 = out["b2_new"]

        if e % 100 == 0 or e == epochs - 1:
            preds = []
            youts = []

            for i in range(4):
                _, _, _, a2, pred = forward(inputs[i], W1, b1, W2, b2)
                preds.append(int(pred.numpy()[0]))
                youts.append(float(a2.numpy()[0]))

            errors = sum(int(preds[i] != int(labels[i,0].numpy())) for i in range(4))

            history.append({
                "epoch": e,
                "errors": errors,
                "W1_00": float(W1[0,0].numpy()),
                "W1_10": float(W1[1,0].numpy()),
                "b1_0": float(b1[0].numpy()),
                "W2_0": float(W2[0].numpy()),
                "b2": float(b2.numpy()),
                "y00": youts[0],
                "y01": youts[1],
                "y10": youts[2],
                "y11": youts[3],
            })

    return W1, b1, W2, b2, pd.DataFrame(history)

In [11]:
W1_f, b1_f, W2_f, b2_f, history_df = train_epochs(
    W1_init, b1_init, W2_init, b2_init,
    epochs=EPOCHS,
    lr=LR
)

history_df[["epoch", "errors"]]

,epoch,errors
0,0,2
1,100,1
2,200,1
3,300,2
4,400,2
...,...,...
56,5600,0
57,5700,0
58,5800,0
59,5900,0


In [12]:
rows = []

for i in range(4):
    _, _, _, a2, pred = forward(inputs[i], W1_f, b1_f, W2_f, b2_f)
    rows.append({
        "sample": i,
        "x1": float(inputs[i,0].numpy()),
        "x2": float(inputs[i,1].numpy()),
        "y_true": float(labels[i,0].numpy()),
        "y_out": float(a2.numpy()[0]),
        "pred": int(pred.numpy()[0]),
        "correct": int(int(pred.numpy()[0]) == int(labels[i,0].numpy()))
    })

final_df = pd.DataFrame(rows)
final_df

,sample,x1,x2,y_true,y_out,pred,correct
0,0,0.0,0.0,0.0,0.069946,0,1
1,1,0.0,1.0,1.0,0.904186,1,1
2,2,1.0,0.0,1.0,0.924966,1,1
3,3,1.0,1.0,0.0,0.120199,0,1


In [13]:
final_weights_df = pd.DataFrame([{
    "W1_00": float(W1_f[0,0].numpy()),
    "W1_10": float(W1_f[1,0].numpy()),
    "b1_0": float(b1_f[0].numpy()),
    "W2_0": float(W2_f[0].numpy()),
    "b2": float(b2_f.numpy())
}]).T.rename(columns={0: "final_value"})

final_weights_df

,final_value
W1_00,2.856678
W1_10,-5.600501
b1_0,-1.027885
W2_0,3.920326
b2,-4.860824


In [14]:
print("Final errors =", 4 - final_df["correct"].sum())

Final errors = 0


In [16]:
import struct

def hex_to_float32(h):
    return struct.unpack('!f', bytes.fromhex(h))[0]

vivado_hex = {
    "W1_00": "4036d402",
    "W1_10": "c0b33751",
    "W2_0":  "407ae6b8",
    "b1_0":  "bf839208",
    "b2":    "c09b8bd5",
}

vivado_ref = {k: hex_to_float32(v) for k, v in vivado_hex.items()}
vivado_ref

{'W1_00': 2.856689929962158,
 'W1_10': -5.6005024909973145,
 'W2_0': 3.9203319549560547,
 'b1_0': -1.0278940200805664,
 'b2': -4.860819339752197}

In [17]:
jupyter_ref = {
    "W1_00": 2.856678,
    "W1_10": -5.600501,
    "W2_0":  3.920326,
    "b1_0": -1.027885,
    "b2":   -4.860824,
}

import pandas as pd

cmp = pd.DataFrame({
    "vivado": vivado_ref,
    "jupyter": jupyter_ref,
})
cmp["difference"] = cmp["jupyter"] - cmp["vivado"]
cmp

,vivado,jupyter,difference
W1_00,2.856690,2.856678,-0.000012
W1_10,-5.600502,-5.600501,0.000001
W2_0,3.920332,3.920326,-0.000006
b1_0,-1.027894,-1.027885,0.000009
b2,-4.860819,-4.860824,-0.000005
